# REKTY — Krea 2 + LoRA REKTY ANJANY di Google Colab (gratis)

Jalankan **base model Krea 2 + LoRA REKTY ANJANY-mu** di GPU gratis Colab, lalu publikasikan lewat Cloudflare Tunnel sebagai endpoint **OpenAI-compatible** (`/v1/images/generations`) — siap didaftarkan ke Pollinations sebagai community model.

```
[Colab T4 GPU]  ComfyUI (Krea 2 + LoRA REKTY ANJANY)
      |  /prompt + /history
      v
gateway.py  ->  POST /v1/images/generations  (format OpenAI)
      |
      v
cloudflared  ->  https://xxxx.trycloudflare.com  (URL publik)
```

**Cara pakai:** jalankan semua sel dari atas ke bawah (Runtime -> Run all), tunggu sampai muncul URL `https://xxx.trycloudflare.com`, lalu pakai URL itu di Pollinations / app REKTY. Total download ~19 GB (sekali saja).

## ⚙️ KONFIGURASI

Jalankan sel berikut, lalu pilih di **Runtime -> Change runtime type**:
- **Hardware accelerator: GPU** (T4 — gratis, cukup untuk Krea 2 fp8)
- Runtime -> **Run all** (atau jalankan sel satu per satu)

In [ ]:
# ============================================================
#  KONFIGURASI  -  ubah sesuai kebutuhan, lalu jalankan sel ini
# ============================================================

# CHECKPOINT: default = checkpoint-mu sendiri (Krea2_by_Rekty_Quantize_00001_) di HuggingFace.
# Kalau mau pakai Krea 2 Turbo resmi, kosongkan CHECKPOINT_URL ("").
CHECKPOINT_URL = "https://huggingface.co/rekty1988/KREA2_BY_REKTY/resolve/main/Krea2_by_Rekty_Quantize_00001_.safetensors"

# LoRA REKTY ANJANY-mu (sudah publik di HuggingFace - dipakai otomatis)
LORA_URL      = "https://huggingface.co/rekty1988/REKTY_ANJANY/resolve/main/rekty%20anjany.safetensors"
LORA_STRENGTH = 0.5

# File pendukung resmi Krea 2 (jangan diubah)
TE_URL  = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"
VAE_URL = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/vae/qwen_image_vae.safetensors"
OFFICIAL_CKPT_URL = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/diffusion_models/krea2_turbo_fp8_scaled.safetensors"

# Parameter generate default (bisa diubah lewat endpoint nanti)
STEPS, CFG = 8, 1.0
SAMPLER, SCHEDULER = "er_sde", "simple"
MODEL_ID = "rekty1988/anjany"

print("Konfigurasi siap. CHECKPOINT :", CHECKPOINT_URL or "Krea 2 Turbo resmi")


In [ ]:
# 1) Cek GPU - harus ada Tesla T4 / P100 / L4 (bukan CPU)
!nvidia-smi


In [ ]:
# 2) Install ComfyUI (sekali saja)
import os
COMFY = "/content/ComfyUI"
if not os.path.exists(COMFY + "/main.py"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}
    !pip install -q -r {COMFY}/requirements.txt
print("ComfyUI siap di", COMFY)


In [ ]:
# 3) Download semua model (~19 GB total, sekitar 5-15 menit)
import os, urllib.request

CKPT_URL = CHECKPOINT_URL or OFFICIAL_CKPT_URL
ckpt_name = os.path.basename(CKPT_URL).split("?")[0]
CKPT_DEST = COMFY + "/models/diffusion_models/" + ckpt_name

def dl(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print("sudah ada :", os.path.basename(dest))
        return
    print("Download  :", os.path.basename(dest))
    urllib.request.urlretrieve(url, dest)
    print("  selesai :", round(os.path.getsize(dest) / 2**30, 2), "GB")

# Checkpoint: coba file-mu dulu; kalau gagal, otomatis pakai Krea 2 Turbo resmi
try:
    dl(CKPT_URL, CKPT_DEST)
except Exception as e:
    print("Gagal download checkpoint-mu:", e)
    print("-> Fallback ke Krea 2 Turbo resmi (publik)")
    CKPT_URL = OFFICIAL_CKPT_URL
    ckpt_name = os.path.basename(CKPT_URL).split("?")[0]
    CKPT_DEST = COMFY + "/models/diffusion_models/" + ckpt_name
    dl(CKPT_URL, CKPT_DEST)

for url, dest in [
    (TE_URL,   COMFY + "/models/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"),
    (VAE_URL,  COMFY + "/models/vae/qwen_image_vae.safetensors"),
    (LORA_URL, COMFY + "/models/loras/rekty anjany.safetensors"),
]:
    dl(url, dest)

print()
print("SEMUA MODEL SIAP. Checkpoint :", ckpt_name)


In [ ]:
# 4) Bangun workflow ComfyUI (API format) - Krea 2 + LoRA REKTY ANJANY
import json

workflow = {
  "1":  {"class_type": "UNETLoader",          "inputs": {"unet_name": ckpt_name, "weight_dtype": "default"}},
  "2":  {"class_type": "LoraLoaderModelOnly", "inputs": {"lora_name": "rekty anjany.safetensors", "strength_model": LORA_STRENGTH, "model": ["1", 0]}},
  "3":  {"class_type": "CLIPLoader",          "inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2"}},
  "4":  {"class_type": "CLIPTextEncode",      "inputs": {"text": "contoh prompt", "clip": ["3", 0]}},
  "5":  {"class_type": "CLIPTextEncode",      "inputs": {"text": "low quality, worst quality, blurry", "clip": ["3", 0]}},
  "6":  {"class_type": "EmptyLatentImage",    "inputs": {"width": 832, "height": 1536, "batch_size": 1}},
  "7":  {"class_type": "KSampler",            "inputs": {"seed": 42, "steps": STEPS, "cfg": CFG, "sampler_name": SAMPLER, "scheduler": SCHEDULER, "denoise": 1.0, "model": ["2", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0]}},
  "8":  {"class_type": "VAEDecode",           "inputs": {"samples": ["7", 0], "vae": ["9", 0]}},
  "9":  {"class_type": "VAELoader",           "inputs": {"vae_name": "qwen_image_vae.safetensors"}},
  "10": {"class_type": "SaveImage",           "inputs": {"filename_prefix": "rekty", "images": ["8", 0]}},
}

with open("/content/workflow_api.json", "w") as f:
    json.dump(workflow, f)
print("workflow_api.json siap -", ckpt_name)


In [ ]:
# 5) Jalankan ComfyUI di background + tunggu sampai siap
import subprocess, sys, time, urllib.request

log = open("/content/comfy.log", "w")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)

ok = False
for _ in range(120):
    time.sleep(3)
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3)
        if r.status == 200:
            ok = True
            break
    except Exception:
        pass

if ok:
    print("ComfyUI SIAP di port 8188 (pid", proc.pid, ")")
else:
    print("Belum siap - lihat /content/comfy.log :")
    print(open("/content/comfy.log").read()[-2000:])


In [ ]:
# 6) Pasang + jalankan gateway OpenAI-compatible (endpoint /v1/images/generations)
import base64, os, subprocess, sys, time, urllib.request

GATEWAY_B64 = "CmltcG9ydCBiYXNlNjQKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgdGltZQppbXBvcnQgdXJsbGliLnBhcnNlCgppbXBvcnQgcmVxdWVzdHMKZnJvbSBmYXN0YXBpIGltcG9ydCBGYXN0QVBJLCBSZXF1ZXN0CmZyb20gZmFzdGFwaS5yZXNwb25zZXMgaW1wb3J0IEpTT05SZXNwb25zZQoKIyAtLS0tLS0tLS0tLS0tLS0tIGtvbmZpZ3VyYXNpIC0tLS0tLS0tLS0tLS0tLS0KQ09NRllfVVJMID0gb3MuZW52aXJvbi5nZXQoIkNPTUZZX1VSTCIsICJodHRwOi8vMTI3LjAuMC4xOjgxODgiKSAgICMgVVJMIENvbWZ5VUkKV09SS0ZMT1dfRklMRSA9IG9zLmVudmlyb24uZ2V0KAogICAgIldPUktGTE9XX0ZJTEUiLAogICAgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSwgIndvcmtmbG93X2FwaS5qc29uIiksCikKTE9SQV9OQU1FID0gb3MuZW52aXJvbi5nZXQoIkxPUkFfTkFNRSIsICJyZWt0eSBhbmphbnkuc2FmZXRlbnNvcnMiKQpBUElfVE9LRU4gPSBvcy5lbnZpcm9uLmdldCgiR0FURVdBWV9UT0tFTiIsICIiKSAgICMgb3BzaW9uYWw6IGthc2loIHRva2VuIHNlbmRpcmkKTU9ERUxfTkFNRVMgPSBbInJla3R5MTk4OC9hbmphbnkiLCAia3JlYTIiLCAiYW5qYW55IiwgInJla3R5Il0KCmFwcCA9IEZhc3RBUEkodGl0bGU9IlJFS1RZIENvbWZ5VUkgR2F0ZXdheSAoT3BlbkFJLWNvbXBhdGlibGUpIikKCgpkZWYgbG9hZF93b3JrZmxvdygpIC0+IGRpY3Q6CiAgICB3aXRoIG9wZW4oV09SS0ZMT1dfRklMRSwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKCgpkZWYgZmluZF9ub2Rlcyh3ZjogZGljdCwgbm9kZV90eXBlOiBzdHIpOgogICAgIiIiU2VtdWEgbm9kZSBDb21meVVJIGRlbmdhbiB0aXBlIHRlcnRlbnR1IChBUEkgZm9ybWF0OiBkaWN0IHtpZDoge2NsYXNzX3R5cGUsIGlucHV0c319KS4iIiIKICAgIG91dCA9IFtdCiAgICBmb3IgbmlkLCBub2RlIGluIHdmLml0ZW1zKCk6CiAgICAgICAgaWYgbm9kZS5nZXQoImNsYXNzX3R5cGUiKSA9PSBub2RlX3R5cGU6CiAgICAgICAgICAgIG91dC5hcHBlbmQoKG5pZCwgbm9kZSkpCiAgICByZXR1cm4gb3V0CgoKZGVmIGluamVjdCh3ZjogZGljdCwgcHJvbXB0OiBzdHIsIHNpemU6IHN0ciwgc2VlZDogaW50LCBzdGVwczogaW50LCBjZmc6IGZsb2F0LAogICAgICAgICAgIHNhbXBsZXI6IHN0ciwgc2NoZWR1bGVyOiBzdHIsIG5lZ2F0aXZlOiBzdHIsIGxvcmFfbmFtZTogc3RyKSAtPiBkaWN0OgogICAgdywgaCA9IChpbnQoeCkgZm9yIHggaW4gc2l6ZS5sb3dlcigpLnNwbGl0KCJ4IikpCiAgICAjIHByb21wdCBwb3NpdGlmOiBDTElQVGV4dEVuY29kZSBwZXJ0YW1hIChiaWFzYW55YSAicG9zaXRpdmUiKQogICAgZW5jcyA9IGZpbmRfbm9kZXMod2YsICJDTElQVGV4dEVuY29kZSIpCiAgICBpZiBlbmNzOgogICAgICAgIGVuY3NbMF1bMV1bImlucHV0cyJdWyJ0ZXh0Il0gPSBwcm9tcHQKICAgICAgICBpZiBsZW4oZW5jcykgPiAxOgogICAgICAgICAgICBlbmNzWzFdWzFdWyJpbnB1dHMiXVsidGV4dCJdID0gbmVnYXRpdmUKICAgICMgdWt1cmFuIGxhdGVuc2kKICAgIGZvciBfbmlkLCBub2RlIGluIGZpbmRfbm9kZXMod2YsICJFbXB0eUxhdGVudEltYWdlIik6CiAgICAgICAgbm9kZVsiaW5wdXRzIl1bIndpZHRoIl0gPSB3CiAgICAgICAgbm9kZVsiaW5wdXRzIl1bImhlaWdodCJdID0gaAogICAgIyBzYW1wbGVyCiAgICBmb3IgX25pZCwgbm9kZSBpbiBmaW5kX25vZGVzKHdmLCAiS1NhbXBsZXIiKToKICAgICAgICBub2RlWyJpbnB1dHMiXVsic2VlZCJdID0gc2VlZAogICAgICAgIG5vZGVbImlucHV0cyJdWyJzdGVwcyJdID0gc3RlcHMKICAgICAgICBub2RlWyJpbnB1dHMiXVsiY2ZnIl0gPSBjZmcKICAgICAgICBpZiBub2RlLmdldCgiaW5wdXRzIiwge30pLmdldCgic2FtcGxlcl9uYW1lIikgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG5vZGVbImlucHV0cyJdWyJzYW1wbGVyX25hbWUiXSA9IHNhbXBsZXIKICAgICAgICBpZiBub2RlLmdldCgiaW5wdXRzIiwge30pLmdldCgic2NoZWR1bGVyIikgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG5vZGVbImlucHV0cyJdWyJzY2hlZHVsZXIiXSA9IHNjaGVkdWxlcgogICAgIyBMb1JBIChMb3JhTG9hZGVyIGJpYXNhIG1hdXB1biBMb3JhTG9hZGVyTW9kZWxPbmx5IHVudHVrIFVORVQtb25seSkKICAgIGxvcmFfbm9kZXMgPSBmaW5kX25vZGVzKHdmLCAiTG9yYUxvYWRlciIpICsgZmluZF9ub2Rlcyh3ZiwgIkxvcmFMb2FkZXJNb2RlbE9ubHkiKQogICAgZm9yIF9uaWQsIG5vZGUgaW4gbG9yYV9ub2RlczoKICAgICAgICBub2RlWyJpbnB1dHMiXVsibG9yYV9uYW1lIl0gPSBsb3JhX25hbWUKICAgIHJldHVybiB3ZgoKCmRlZiBydW5fY29tZnkod2Y6IGRpY3QsIHRpbWVvdXQ6IGludCA9IDYwMCkgLT4gbGlzdDoKICAgICIiIktpcmltIHdvcmtmbG93IGtlIENvbWZ5VUksIHR1bmdndSBzZWxlc2FpLCBiYWxhcyBkYWZ0YXIgYjY0IGltYWdlLiIiIgogICAgciA9IHJlcXVlc3RzLnBvc3QoZiJ7Q09NRllfVVJMfS9wcm9tcHQiLCBqc29uPXsicHJvbXB0Ijogd2Z9LCB0aW1lb3V0PTMwKQogICAgci5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgIHByb21wdF9pZCA9IHIuanNvbigpWyJwcm9tcHRfaWQiXQogICAgIyBwb2xsIGhpc3RvcnkKICAgIGRlYWRsaW5lID0gdGltZS50aW1lKCkgKyB0aW1lb3V0CiAgICB3aGlsZSB0aW1lLnRpbWUoKSA8IGRlYWRsaW5lOgogICAgICAgIHRpbWUuc2xlZXAoMS41KQogICAgICAgIGggPSByZXF1ZXN0cy5nZXQoZiJ7Q09NRllfVVJMfS9oaXN0b3J5L3twcm9tcHRfaWR9IiwgdGltZW91dD0zMCkKICAgICAgICBpZiBoLnN0YXR1c19jb2RlICE9IDIwMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkYXRhID0gaC5qc29uKCkKICAgICAgICBpZiBwcm9tcHRfaWQgbm90IGluIGRhdGE6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgb3V0cHV0cyA9IGRhdGFbcHJvbXB0X2lkXS5nZXQoIm91dHB1dHMiLCB7fSkKICAgICAgICBpbWFnZXMgPSBbXQogICAgICAgIGZvciBfbmlkLCBvdXQgaW4gb3V0cHV0cy5pdGVtcygpOgogICAgICAgICAgICBmb3IgaW1nIGluIG91dC5nZXQoImltYWdlcyIsIFtdKToKICAgICAgICAgICAgICAgIGZuID0gaW1nWyJmaWxlbmFtZSJdCiAgICAgICAgICAgICAgICBzdWIgPSBpbWcuZ2V0KCJzdWJmb2xkZXIiLCAiIikKICAgICAgICAgICAgICAgIGN0ID0gaW1nLmdldCgidHlwZSIsICJvdXRwdXQiKQogICAgICAgICAgICAgICAgcSA9IHVybGxpYi5wYXJzZS51cmxlbmNvZGUoeyJmaWxlbmFtZSI6IGZuLCAic3ViZm9sZGVyIjogc3ViLCAidHlwZSI6IGN0fSkKICAgICAgICAgICAgICAgIHJhdyA9IHJlcXVlc3RzLmdldChmIntDT01GWV9VUkx9L3ZpZXc/e3F9IiwgdGltZW91dD02MCkKICAgICAgICAgICAgICAgIHJhdy5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAgICAgICAgIGltYWdlcy5hcHBlbmQoYmFzZTY0LmI2NGVuY29kZShyYXcuY29udGVudCkuZGVjb2RlKCJhc2NpaSIpKQogICAgICAgIGlmIGltYWdlczoKICAgICAgICAgICAgcmV0dXJuIGltYWdlcwogICAgICAgIGlmIGRhdGFbcHJvbXB0X2lkXS5nZXQoInN0YXR1cyIsIHt9KS5nZXQoInN0YXR1c19zdHIiKSA9PSAiZXJyb3IiOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvbWZ5VUkgd29ya2Zsb3cgZXJyb3IiKQogICAgcmFpc2UgVGltZW91dEVycm9yKCJDb21meVVJIHRpZGFrIHNlbGVzYWkgZGFsYW0gYmF0YXMgd2FrdHUiKQoKCkBhcHAucG9zdCgiL3YxL2ltYWdlcy9nZW5lcmF0aW9ucyIpCmFzeW5jIGRlZiBpbWFnZXNfZ2VuZXJhdGlvbnMocmVxOiBSZXF1ZXN0KToKICAgIHRyeToKICAgICAgICBib2R5ID0gYXdhaXQgcmVxLmpzb24oKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKHsiZXJyb3IiOiB7Im1lc3NhZ2UiOiAiSlNPTiB0aWRhayB2YWxpZCIsICJ0eXBlIjogImludmFsaWRfcmVxdWVzdF9lcnJvciJ9fSwgc3RhdHVzX2NvZGU9NDAwKQoKICAgICMgYXV0aCBvcHNpb25hbCAodG9rZW4gc2VuZGlyaSkKICAgIGlmIEFQSV9UT0tFTjoKICAgICAgICBhdXRoID0gcmVxLmhlYWRlcnMuZ2V0KCJhdXRob3JpemF0aW9uIiwgIiIpLnJlcGxhY2UoIkJlYXJlciAiLCAiIikKICAgICAgICBpZiBhdXRoICE9IEFQSV9UT0tFTjoKICAgICAgICAgICAgcmV0dXJuIEpTT05SZXNwb25zZSh7ImVycm9yIjogeyJtZXNzYWdlIjogIlRva2VuIHNhbGFoIiwgInR5cGUiOiAiaW52YWxpZF9hcGlfa2V5In19LCBzdGF0dXNfY29kZT00MDEpCgogICAgbW9kZWwgPSBzdHIoYm9keS5nZXQoIm1vZGVsIiwgInJla3R5MTk4OC9hbmphbnkiKSkKICAgIGlmIG1vZGVsIG5vdCBpbiBNT0RFTF9OQU1FUzoKICAgICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKHsiZXJyb3IiOiB7Im1lc3NhZ2UiOiBmIk1vZGVsICd7bW9kZWx9JyB0aWRhayBkaWtlbmFsIiwgInR5cGUiOiAiaW52YWxpZF9yZXF1ZXN0X2Vycm9yIn19LCBzdGF0dXNfY29kZT00MDApCiAgICBwcm9tcHQgPSBzdHIoYm9keS5nZXQoInByb21wdCIsICIiKSkuc3RyaXAoKQogICAgaWYgbm90IHByb21wdDoKICAgICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKHsiZXJyb3IiOiB7Im1lc3NhZ2UiOiAicHJvbXB0IGtvc29uZyIsICJ0eXBlIjogImludmFsaWRfcmVxdWVzdF9lcnJvciJ9fSwgc3RhdHVzX2NvZGU9NDAwKQogICAgc2l6ZSA9IHN0cihib2R5LmdldCgic2l6ZSIsICI4MzJ4MTUzNiIpKQogICAgbiA9IGludChib2R5LmdldCgibiIsIDEpIG9yIDEpCiAgICBzZWVkID0gaW50KGJvZHkuZ2V0KCJzZWVkIiwgMCkgb3IgMCkKICAgIHN0ZXBzID0gaW50KGJvZHkuZ2V0KCJzdGVwcyIsIDI1KSBvciAyNSkKICAgIGNmZyA9IGZsb2F0KGJvZHkuZ2V0KCJjZmciLCAxLjApIG9yIDEuMCkKICAgIHNhbXBsZXIgPSBzdHIoYm9keS5nZXQoInNhbXBsZXIiLCAiZXJfc2RlIikpCiAgICBzY2hlZHVsZXIgPSBzdHIoYm9keS5nZXQoInNjaGVkdWxlciIsICJzaW1wbGUiKSkKICAgIG5lZ2F0aXZlID0gc3RyKGJvZHkuZ2V0KCJuZWdhdGl2ZV9wcm9tcHQiLCAibG93IHF1YWxpdHksIHdvcnN0IHF1YWxpdHkiKSkKCiAgICB0cnk6CiAgICAgICAgd2YgPSBsb2FkX3dvcmtmbG93KCkKICAgICAgICBpbmplY3Qod2YsIHByb21wdCwgc2l6ZSwgc2VlZCwgc3RlcHMsIGNmZywgc2FtcGxlciwgc2NoZWR1bGVyLCBuZWdhdGl2ZSwgTE9SQV9OQU1FKQogICAgICAgIGltZ3MgPSBydW5fY29tZnkod2YpCiAgICAgICAgIyBwYWthaSBnYW1iYXIgcGVydGFtYSBuIGthbGkgYmlsYSBkaW1pbnRhIGxlYmloIChhdGF1IGtpcmltIHNlc3VhaSBuKQogICAgICAgIG91dCA9IFt7ImI2NF9qc29uIjogaW1nc1tpICUgbGVuKGltZ3MpXX0gZm9yIGkgaW4gcmFuZ2UobildCiAgICAgICAgcmV0dXJuIHsiY3JlYXRlZCI6IGludCh0aW1lLnRpbWUoKSksICJkYXRhIjogb3V0fQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHJldHVybiBKU09OUmVzcG9uc2UoeyJlcnJvciI6IHsibWVzc2FnZSI6IHN0cihlKSwgInR5cGUiOiAic2VydmVyX2Vycm9yIn19LCBzdGF0dXNfY29kZT01MDApCgoKQGFwcC5nZXQoIi92MS9tb2RlbHMiKQphc3luYyBkZWYgbW9kZWxzKCk6CiAgICByZXR1cm4geyJvYmplY3QiOiAibGlzdCIsICJkYXRhIjogW3siaWQiOiBtLCAib2JqZWN0IjogIm1vZGVsIiwgImNyZWF0ZWQiOiAwLCAib3duZWRfYnkiOiAicmVrdHkifSBmb3IgbSBpbiBNT0RFTF9OQU1FU119CgoKQGFwcC5nZXQoIi9oZWFsdGgiKQphc3luYyBkZWYgaGVhbHRoKCk6CiAgICByZXR1cm4geyJvayI6IFRydWUsICJjb21meSI6IENPTUZZX1VSTH0KCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaW1wb3J0IHV2aWNvcm4KICAgIHV2aWNvcm4ucnVuKGFwcCwgaG9zdD0iMC4wLjAuMCIsIHBvcnQ9aW50KG9zLmVudmlyb24uZ2V0KCJQT1JUIiwgODAwMCkpKQo="

open("/content/gateway.py", "w").write(base64.b64decode(GATEWAY_B64).decode("utf-8"))
!pip install -q fastapi uvicorn requests

env = dict(os.environ,
           WORKFLOW_FILE="/content/workflow_api.json",
           LORA_NAME="rekty anjany.safetensors")

glog = open("/content/gateway.log", "w")
gproc = subprocess.Popen([sys.executable, "/content/gateway.py"],
                         stdout=glog, stderr=subprocess.STDOUT, env=env)

ok = False
for _ in range(60):
    time.sleep(2)
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
        if r.status == 200:
            ok = True
            break
    except Exception:
        pass

if ok:
    print("GATEWAY SIAP di port 8000 (pid", gproc.pid, ")")
else:
    print("Belum siap - lihat /content/gateway.log :")
    print(open("/content/gateway.log").read()[-2000:])


In [ ]:
# 7) Publikasikan dengan Cloudflare Tunnel (gratis) - cetak URL publik
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import re, subprocess, time

tlog = open("/content/tunnel.log", "w")
tproc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tlog, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(2)
    txt = open("/content/tunnel.log").read()
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break

print()
print("============================================================")
print("  ENDPOINT PUBLIK KAMU :", url or "(belum dapat - lihat /content/tunnel.log)")
print("============================================================")
if url:
    print("Tes cepat  :")
    print("  curl", url + "/v1/models")
    print("  curl -X POST", url + "/v1/images/generations",
          "-H 'Content-Type: application/json'",
          "-d '{\"prompt\":\"a cat with blue eyes\",\"size\":\"832x1536\"}'")


## ✅ Selesai — cara pakai URL-nya

1. **Salin URL** `https://xxx.trycloudflare.com` dari output sel terakhir (biarkan sel itu tetap berjalan).
2. **Tes langsung** (boleh di sel baru / terminal lokal):
   ```
   curl https://xxx.trycloudflare.com/v1/models
   curl -X POST https://xxx.trycloudflare.com/v1/images/generations \
     -H "Content-Type: application/json" \
     -d '{"model":"rekty1988/anjany","prompt":"a cat with blue eyes","size":"832x1536"}'
   ```
3. **Daftarkan ke Pollinations** (biar muncul di katalog): login **enter.pollinations.ai** dengan akun GitHub → ajukan akses publisher lewat template issue **"Community Model Publisher Allowlist"** → setelah disetujui, daftarkan di **my-models**:
   - Nama model : `rekty1988/anjany`
   - Endpoint   : `https://xxx.trycloudflare.com/v1`
4. Model muncul di katalog Pollinations → otomatis terlihat di **dropdown model Pollinations** di aplikasi REKTY.

### ⚠️ Penting (batas Colab gratis)
- Sesi Colab gratis **mati otomatis**: idle ~90 menit atau maksimal ~12 jam. Saat mati, URL ikut mati — jalankan ulang semua sel (model tersimpan di disk, download tidak perlu diulang).
- Untuk **24/7** (syarat Pollinations community model) butuh VPS GPU berbayar — lihat `selfhost/README.md`.
- Mau pakai **checkpoint-mu sendiri** (`Krea2_by_Rekty_Quantize_00001_`)? Upload dulu ke HuggingFace (public), tempel URL-nya di sel KONFIGURASI (`CHECKPOINT_URL`), lalu jalankan ulang dari sel download.